# 14.02 · Speech: STT & TTS

**Chapter:** Multimodal AI · **Milestone:** M7

## Goals
- [ ] Audio basics: sample rate, waveform, log-mel spectrogram (what Whisper actually sees)
- [ ] Speech-to-text: Whisper sizes, local engines (mlx-whisper, faster-whisper, whisper.cpp) vs APIs (Groq, Deepgram, AssemblyAI, OpenAI)
- [ ] Measure **WER** (word error rate) and **RTF** (real-time factor); test robustness to noise and accents/languages
- [ ] Text-to-speech: open (Kokoro) vs paid (ElevenLabs, OpenAI, Cartesia); evaluate TTS with an STT round-trip
- [ ] Transcribe a meeting and summarise it into action items

## Resources
- [HF Audio Course](https://huggingface.co/learn/audio-course) — free
- [Whisper paper](https://arxiv.org/abs/2212.04356) · [faster-whisper](https://github.com/SYSTRAN/faster-whisper) · [mlx-whisper](https://github.com/ml-explore/mlx-examples/tree/main/whisper) — free
- [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M) · [TTS Arena leaderboard](https://huggingface.co/spaces/TTS-AGI/TTS-Arena-V2) — free
- [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard) · [Groq speech-to-text](https://console.groq.com/docs/speech-to-text) — free tier

Audio is just a 1-D array of samples (16,000 per second for speech models). Whisper turns 30-second windows into an 80-band **log-mel
spectrogram** (an image of frequency energy over time), encodes it with a transformer, and a decoder writes the text token by token —
an encoder-decoder LLM whose "prompt" is sound.

In [ ]:
import re
import subprocess
import time

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

from aieng import DATA
from aieng.llm import available, chat, client

AUDIO = DATA / "audio"
AUDIO.mkdir(exist_ok=True)
SR = 16_000

## 1. A test set with exact transcripts, for free
macOS `say` speaks any text with many voices, so we get audio with known ground truth (on Linux: `espeak-ng` or the TTS in §5).
Synthetic speech is cleaner than real recordings — treat these WERs as optimistic and test on real audio before trusting a model.

In [ ]:
def say(text, voice, path):
    subprocess.run(["say", "-v", voice, "-o", str(path), f"--data-format=LEI16@{SR}", text], check=True)
    audio, sr = sf.read(path, dtype="float32")
    assert sr == SR
    return audio


SENTENCES = [
    "The quarterly revenue grew by twelve percent compared to last year.",
    "Please schedule the meeting with Doctor Ramirez for Thursday at three thirty.",
    "Kubernetes autoscaling depends on the metrics server and the horizontal pod autoscaler.",
    "We shipped the retrieval augmented generation feature to production on Monday.",
    "The invoice number is four five seven two, and the total is eighty dollars.",
    "Fine tuning a small model can beat a large one on a narrow task.",
    "Remember to rotate the API keys before the security audit next week.",
    "Latency at the ninety fifth percentile went from two seconds to eight hundred milliseconds.",
]
VOICES = ["Daniel", "Flo (English (US))"]
clips = [(say(t, v, AUDIO / f"clip{i}-{j}.wav"), t, v) for i, t in enumerate(SENTENCES) for j, v in enumerate(VOICES)]
print(len(clips), "clips,", f"{sum(len(a) for a, *_ in clips) / SR:.0f}s of audio")
display(Audio(clips[0][0], rate=SR))

## 2. What the model sees

In [ ]:
from transformers import WhisperFeatureExtractor

audio = clips[2][0]
mel = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")(audio, sampling_rate=SR).input_features[0]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 4))
a1.plot(np.arange(len(audio)) / SR, audio, lw=0.4); a1.set_xlim(0, len(audio) / SR); a1.set_title("waveform"); a1.set_xlabel("s")
a2.imshow(mel[:, : int(len(audio) / SR * 100)], origin="lower", aspect="auto", cmap="magma"); a2.set_title("log-mel spectrogram (80 bands × 10 ms frames)")
plt.tight_layout(); plt.show()

## 3. Speech-to-text: WER and speed
**WER** = (substitutions + deletions + insertions) / words in the reference, after normalising case and punctuation.
**RTF** = processing time / audio duration (0.05 = 20× faster than real time).

Normalisation matters as much as the model: "twelve percent" vs "12%" is a formatting difference, not a recognition error. We report both a
naive WER (lowercase, no punctuation) and WER after **Whisper's English normaliser** (spelled numbers → digits, British → American spelling…),
which is what leaderboards use.

In [ ]:
from transformers import WhisperTokenizer

_wn = WhisperTokenizer.from_pretrained("openai/whisper-tiny")


def naive(s):
    return " ".join(re.sub(r"[^\w\s']", " ", s.lower()).split()) or "<empty>"


def normalize(s):
    return _wn.normalize(s).strip() or "<empty>"


def wer(ref, hyp):
    return jiwer.wer(normalize(ref), normalize(hyp))


def benchmark(name, transcribe, data=clips):
    transcribe(data[0][0])  # warm-up (model load)
    start, refs, hyps = time.perf_counter(), [], []
    for audio, text, _ in data:
        hyps.append(transcribe(audio)); refs.append(text)
    secs = time.perf_counter() - start
    w = jiwer.wer([normalize(r) for r in refs], [normalize(h) for h in hyps])
    w_naive = jiwer.wer([naive(r) for r in refs], [naive(h) for h in hyps])
    rtf = secs / (sum(len(a) for a, *_ in data) / SR)
    print(f"{name:34} WER {w:6.1%} (naive {w_naive:6.1%}) | RTF {rtf:.3f}")
    return w, hyps


import mlx_whisper
import torch
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else "cpu"
tiny = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en", device=device)

results = {}
results["whisper-tiny.en (HF, 39M)"] = benchmark("whisper-tiny.en (HF, 39M)", lambda a: tiny(a.copy())["text"])
for repo in ["mlx-community/whisper-small-mlx", "mlx-community/whisper-large-v3-turbo"]:
    name = repo.split("/")[1] + " (MLX)"
    results[name] = benchmark(name, lambda a, r=repo: mlx_whisper.transcribe(a, path_or_hf_repo=r, language="en")["text"])

APIs: Groq serves Whisper large-v3-turbo on its free tier through the OpenAI-compatible `audio.transcriptions` endpoint
(OpenAI, Deepgram, AssemblyAI, ElevenLabs Scribe and Gemini work similarly; Deepgram/AssemblyAI add diarization and streaming).

In [ ]:
if "groq" in available():
    gc, _ = client("groq")

    def groq_stt(audio):
        path = AUDIO / "tmp.wav"; sf.write(path, audio, SR)
        with open(path, "rb") as f:
            return gc.audio.transcriptions.create(model="whisper-large-v3-turbo", file=f, language="en").text

    results["groq whisper-large-v3-turbo"] = benchmark("groq whisper-large-v3-turbo (API)", groq_stt)

for name, (w, hyps) in results.items():
    errs = [(normalize(t), normalize(h)) for (_, t, _), h in zip(clips, hyps) if wer(t, h) > 0]
    if errs:
        print(f"\n{name}: e.g. REF {errs[0][0]!r}\n{'':{len(name)}}       HYP {errs[0][1].strip()!r}")

Compare the two WER columns: most naive "errors" were number formatting. What remains is names and jargon. Fix those by passing domain
words as a **prompt** (`initial_prompt="Kubernetes, RAG, Ramirez"` in mlx-whisper / `prompt=` in the API) to bias the decoder.

## 4. Robustness: noise and language

In [ ]:
def add_noise(audio, snr_db, seed=0):
    noise = np.random.default_rng(seed).normal(size=audio.shape).astype(np.float32)
    scale = np.sqrt(np.mean(audio ** 2) / (np.mean(noise ** 2) * 10 ** (snr_db / 10)))
    return audio + scale * noise


turbo = lambda a: mlx_whisper.transcribe(a, path_or_hf_repo="mlx-community/whisper-large-v3-turbo", language="en")["text"]
for snr in (20, 5, 0):
    noisy = [(add_noise(a, snr), t, v) for a, t, v in clips]
    benchmark(f"turbo @ SNR {snr:2} dB", turbo, noisy)
    benchmark(f"tiny.en @ SNR {snr:2} dB", lambda a: tiny(a.copy())["text"], noisy)
display(Audio(add_noise(clips[0][0], 0), rate=SR))

Bigger models degrade far more gracefully. A WER above 100% means the model wrote *more* words than were spoken: small Whisper models
**hallucinate** (repeat phrases, invent text) on noise and silence. Real-world noise (music, cross-talk, phone codecs) is harder than white noise — add a
**VAD** (voice activity detection, e.g. Silero VAD) to skip silence and reduce hallucinated text on non-speech.

Whisper is multilingual and detects the language itself:

In [ ]:
es = "Buenos días. La reunión de mañana se mueve a las diez y media porque el cliente llega tarde."
es_audio = say(es, "Eddy (Spanish (Mexico))", AUDIO / "es.wav")
r = mlx_whisper.transcribe(es_audio, path_or_hf_repo="mlx-community/whisper-large-v3-turbo")
print(f"detected {r['language']!r}: {r['text'].strip()}\nWER {wer(es, r['text']):.1%}")

## 5. Transcribe a meeting and summarise it
A three-person stand-up, each person a different voice, glued together with pauses. Whisper returns **segments** with timestamps;
it does *not* know who spoke (that's **diarization**: pyannote, WhisperX, or API features).

In [ ]:
MEETING = [
    ("Daniel", "Morning everyone. Quick stand-up. Ana, how is the evaluation pipeline going?"),
    ("Flo (English (US))", "It's running in CI now. Two tests fail on the citation check, I'll fix them by Wednesday."),
    ("Daniel", "Great. Bruno, what about the deployment?"),
    ("Eddy (English (US))", "The space is live, but cold starts take forty seconds. I want to bake the index into the image."),
    ("Daniel", "Do it. And let's move the demo with the client to Friday at ten. I'll send the invite."),
    ("Flo (English (US))", "One more thing: the Gemini key expires next week, someone needs to rotate it."),
    ("Eddy (English (US))", "I'll take that."),
]
pause = np.zeros(int(0.6 * SR), dtype=np.float32)
meeting = np.concatenate([np.concatenate([say(t, v, AUDIO / f"m{i}.wav"), pause]) for i, (v, t) in enumerate(MEETING)])
sf.write(AUDIO / "meeting.wav", meeting, SR)
print(f"meeting: {len(meeting) / SR:.0f}s")

r = mlx_whisper.transcribe(meeting, path_or_hf_repo="mlx-community/whisper-large-v3-turbo", language="en")
transcript = "\n".join(f"[{s['start']:5.1f}s] {s['text'].strip()}" for s in r["segments"])
print(transcript)

In [ ]:
summary = chat("Summarise this stand-up meeting transcript in 2 sentences, then list action items as '- owner (if known): task — deadline'. "
               "Speakers aren't labelled; infer owners from context only when clear.\n\n" + transcript, temperature=0)
print(summary)

Check the owners against the script: without speaker labels the model can only guess who said "I'll take that" — wrong owners in
action items are the classic failure of transcript summaries. Diarization (exercise 4) or a meeting tool that knows participants fixes it.

## 6. Text-to-speech
**Kokoro-82M** is a small open TTS model with near-commercial quality (Apache-2.0), ~24 kHz output, several voices and languages.
Evaluate TTS two ways: **listen** (MOS-style human ratings, or arenas) and an automatic **intelligibility** check — transcribe the TTS output
with Whisper and compute WER against the input text.

In [ ]:
from kokoro import KPipeline

tts = KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M")  # "a" = American English, "b" = British


def kokoro(text, voice="af_heart"):
    return np.concatenate([a.numpy() if hasattr(a, "numpy") else a for _, _, a in tts(text, voice=voice)])


kokoro("warm up")
for voice in ("af_heart", "am_michael", "bf_emma"):
    start = time.perf_counter()
    speech = kokoro(SENTENCES[3], voice)
    secs = time.perf_counter() - start
    sf.write(AUDIO / f"kokoro-{voice}.wav", speech, 24_000)
    resampled = np.interp(np.arange(0, len(speech), 24_000 / SR), np.arange(len(speech)), speech).astype(np.float32)  # 24k -> 16k
    print(f"{voice:10} RTF {secs / (len(speech) / 24_000):.3f} | round-trip WER {wer(SENTENCES[3], turbo(resampled)):.1%}")
display(Audio(str(AUDIO / "kokoro-af_heart.wav")))

| | Open / local | API |
|---|---|---|
| **STT** | Whisper (mlx-whisper, faster-whisper, whisper.cpp), NVIDIA Parakeet/Canary, Moonshine (edge), Voxtral | Groq, Deepgram, AssemblyAI, OpenAI, ElevenLabs Scribe, Gemini |
| **TTS** | Kokoro, Piper (tiny, CPU), F5-TTS / XTTS (voice cloning), Chatterbox, Dia, Sesame CSM | ElevenLabs, OpenAI `gpt-4o-mini-tts`, Cartesia, Gemini TTS, Azure |

What matters in production: streaming (partial transcripts / first audio chunk in <300 ms, 14.03), diarization, word timestamps,
custom vocabulary, languages you need, and **consent** — never clone a voice without the speaker's permission.

## Exercises
1. Transcribe a real meeting recording of your own (or a podcast episode) locally and summarise it; compare with the Groq API on WER (label 2
   minutes by hand) and speed.
2. Record yourself reading the 8 sentences (QuickTime → File → New Audio Recording, export, convert to 16 kHz with
   `ffmpeg -i in.m4a -ar 16000 -ac 1 out.wav`). How do WERs on your real voice compare with the synthetic ones?
3. Pass `initial_prompt="Kubernetes, Ramirez, retrieval augmented generation"` to mlx-whisper and measure the WER change on the jargon sentences.
4. Add speaker labels to the meeting with diarization (pyannote needs a free HF token; or use the known pauses between turns as a cheap heuristic).

In [ ]:
# your code here